# Oxta Contábil — Smoke Train

**Objetivo:** Verificar end-to-end que conseguimos treinar um modelo nos dados que baixamos do BR-TaxQA-R (478 Q&A IRPF + acórdãos CARF). Não é treino de produção — é um sanity check pra confirmar que:

1. Os dados carregam e fazem sentido
2. A formatação ChatML está correta
3. Um modelo PT-BR pequeno consegue aprender com eles (loss desce)
4. Geração depois do treino é melhor do que antes (mesmo que ainda imperfeita)

**Modelo usado:** `pierreguillou/gpt2-small-portuguese` (125M, GPT-2 small em PT-BR). Pequeno o suficiente pra treinar em ~10 min num T4 grátis do Colab, grande o suficiente pra mostrar diferença antes/depois.

**Importante:** este NÃO é o modelo final do Oxta Contábil. O modelo final será o runtime C++ NSOS treinado em escala 400M-1B. Este notebook só prova que o pipeline funciona com os dados que temos.

**Tempo estimado:** ~10-15 min em GPU (T4), ~45-60 min em CPU.


## 1. Setup — dependências e GPU


In [ ]:
# Install deps (Colab geralmente já tem torch + transformers, mas garante versões compatíveis)
!pip install -q 'transformers>=4.40' 'datasets>=2.16' 'accelerate>=0.27' 'matplotlib' 'huggingface_hub' 'tqdm' 2>&1 | tail -5


In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB')

# Seed everything pra resultados reprodutíveis
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)


## 2. Localizar dataset BR-TaxQA-R

Se você já rodou `download_datasets.py`, os arquivos estão em `artifacts/data/oxta_contabil/raw/br_taxqa/`.

Se está no Colab e não tem os dados, esta célula baixa direto do HuggingFace.


In [ ]:
# Localiza ou baixa o dataset
LOCAL_CANDIDATES = [
    Path('artifacts/data/oxta_contabil/raw/br_taxqa'),
    Path('../artifacts/data/oxta_contabil/raw/br_taxqa'),
    Path('/content/br_taxqa'),  # Colab default
]

dataset_dir = None
for candidate in LOCAL_CANDIDATES:
    if candidate.exists() and any(candidate.glob('questions_QA_*.json')):
        dataset_dir = candidate
        print(f'Encontrado dataset local: {dataset_dir}')
        break

if dataset_dir is None:
    print('Não achei localmente. Baixando do HuggingFace...')
    from huggingface_hub import snapshot_download
    dataset_dir = Path('/content/br_taxqa') if 'google.colab' in sys.modules else Path('./br_taxqa')
    dataset_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='unicamp-dl/BR-TaxQA-R',
        repo_type='dataset',
        local_dir=str(dataset_dir),
        local_dir_use_symlinks=False,
    )
    print(f'Baixado em: {dataset_dir}')

for f in sorted(dataset_dir.glob('*.json')):
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name}: {size_mb:.2f} MB')


## 3. Carregar e explorar os dados


In [ ]:
# Acha o arquivo de questions automaticamente (versão pode mudar)
questions_file = sorted(dataset_dir.glob('questions_QA_*.json'))[-1]
print(f'Carregando: {questions_file.name}')

with open(questions_file, encoding='utf-8') as f:
    raw_data = json.load(f)

# Estrutura do BR-TaxQA-R: dict com chave 'questions' OU lista direta
if isinstance(raw_data, dict) and 'questions' in raw_data:
    questions = raw_data['questions']
elif isinstance(raw_data, list):
    questions = raw_data
else:
    questions = [v for v in raw_data.values() if isinstance(v, dict)]

print(f'Total de Q&A pairs: {len(questions)}')
print(f'Chaves do primeiro item: {list(questions[0].keys())[:10]}')


In [ ]:
# Mostra 3 exemplos pra entender o formato
for i, q in enumerate(questions[:3]):
    print(f'\n{"=" * 70}')
    print(f'Q&A #{i + 1}')
    print("=" * 70)
    # Os campos variam por versão; tentamos os mais comuns
    pergunta = q.get('question_text', q.get('question', '(sem pergunta)'))
    resposta = q.get('answer', q.get('answer_cleaned', '(sem resposta)'))
    if isinstance(resposta, list):
        resposta = '\n'.join(resposta)
    print(f'PERGUNTA:\n  {pergunta[:300]}...' if len(str(pergunta)) > 300 else f'PERGUNTA:\n  {pergunta}')
    print(f'\nRESPOSTA:\n  {str(resposta)[:500]}...' if len(str(resposta)) > 500 else f'\nRESPOSTA:\n  {resposta}')


In [ ]:
# Estatísticas: distribuição de comprimentos
import statistics

pergunta_lens = []
resposta_lens = []
for q in questions:
    p = q.get('question_text', q.get('question', ''))
    r = q.get('answer', q.get('answer_cleaned', ''))
    if isinstance(r, list):
        r = '\n'.join(str(x) for x in r)
    pergunta_lens.append(len(str(p)))
    resposta_lens.append(len(str(r)))

print(f'Pergunta: min={min(pergunta_lens)} max={max(pergunta_lens)} mediana={int(statistics.median(pergunta_lens))} chars')
print(f'Resposta: min={min(resposta_lens)} max={max(resposta_lens)} mediana={int(statistics.median(resposta_lens))} chars')


## 4. Formatação ChatML

Convertemos cada Q&A num exemplo de treino no formato ChatML — o mesmo que vamos usar no Oxta runtime depois:

```
<|im_start|>system
Você é Oxta Contábil, assistente fiscal brasileiro...
<|im_end|>
<|im_start|>user
{pergunta}
<|im_end|>
<|im_start|>assistant
{resposta}
<|im_end|>
```


In [ ]:
SYSTEM_PROMPT = (
    'Você é Oxta Contábil, um assistente fiscal brasileiro construído do zero. '
    'Responda perguntas sobre Imposto de Renda, contabilidade e legislação fiscal '
    'do Brasil de forma clara, citando a fonte quando possível.'
)

def to_chatml(pergunta: str, resposta: str) -> str:
    return (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{pergunta.strip()}<|im_end|>\n'
        f'<|im_start|>assistant\n{resposta.strip()}<|im_end|>'
    )

# Constrói o dataset formatado
formatted = []
for q in questions:
    p = q.get('question_text', q.get('question', ''))
    r = q.get('answer', q.get('answer_cleaned', ''))
    if isinstance(r, list):
        r = '\n'.join(str(x) for x in r)
    if not p or not r:
        continue
    formatted.append(to_chatml(str(p), str(r)))

print(f'Exemplos formatados: {len(formatted)}')
print(f'\nExemplo formatado (primeiros 600 chars):')
print('-' * 70)
print(formatted[0][:600])


## 5. Tokenizer e modelo


In [ ]:
MODEL_NAME = 'pierreguillou/gpt2-small-portuguese'

print(f'Carregando tokenizer {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# GPT-2 não tem pad_token por default; usamos eos como pad
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Adiciona os tokens especiais ChatML — o modelo vai aprender a usá-los
special = ['<|im_start|>', '<|im_end|>']
added = tokenizer.add_special_tokens({'additional_special_tokens': special})
print(f'Tokens especiais adicionados: {added}')

print(f'Vocab size: {len(tokenizer)}')
print(f'Pad token:  {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})')
print(f'EOS token:  {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})')


In [ ]:
# Distribuição de comprimentos em tokens (não chars) — define o max_length de treino
lens = [len(tokenizer.encode(text)) for text in formatted]
print(f'Tokens por exemplo: min={min(lens)} max={max(lens)} mediana={int(statistics.median(lens))}')
print(f'Total de tokens no dataset: {sum(lens):,}')

# Histograma
plt.figure(figsize=(9, 3))
plt.hist(lens, bins=40)
plt.axvline(512, color='r', linestyle='--', label='max_length=512')
plt.axvline(1024, color='orange', linestyle='--', label='max_length=1024')
plt.xlabel('tokens por exemplo')
plt.ylabel('frequência')
plt.title('Distribuição de tamanho dos exemplos BR-TaxQA-R em tokens')
plt.legend()
plt.tight_layout()
plt.show()

# Escolhemos max_length que cabe ~80% dos exemplos sem truncar
lens_sorted = sorted(lens)
p80 = lens_sorted[int(len(lens_sorted) * 0.8)]
MAX_LEN = min(1024, max(256, p80))
print(f'\nUsando max_length={MAX_LEN} (cobre 80% dos exemplos sem truncar)')


In [ ]:
print(f'Carregando modelo {MODEL_NAME}...')
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))  # acomoda os 2 tokens novos
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parâmetros totais:   {total_params:,} ({total_params / 1e6:.1f}M)')
print(f'Parâmetros treinaveis: {trainable:,}')


## 6. Geração ANTES do treino

Vamos perguntar 3 coisas sobre IRPF ao modelo cru. Esperamos respostas genéricas/erradas — esta é a baseline pra comparar depois do fine-tune.


In [ ]:
TEST_QUESTIONS = [
    'Quem é obrigado a declarar Imposto de Renda?',
    'Qual o prazo para entregar a declaração do IR em 2024?',
    'Posso deduzir despesas com plano de saúde no IR?',
]

def generate_answer(model, tokenizer, question: str, max_new_tokens: int = 150) -> str:
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
        )
    full = tokenizer.decode(out[0], skip_special_tokens=False)
    # Pega só a parte do assistant
    answer = full.split('<|im_start|>assistant\n', 1)[-1]
    answer = answer.split('<|im_end|>', 1)[0].strip()
    return answer

model.eval()
before_answers = {}
for q in TEST_QUESTIONS:
    print(f'\nPERGUNTA: {q}')
    print('-' * 70)
    ans = generate_answer(model, tokenizer, q)
    before_answers[q] = ans
    print(f'RESPOSTA (antes do treino):\n  {ans[:400]}')


## 7. Dataset class e DataLoader


In [ ]:
class ChatMLDataset(Dataset):
    """Tokeniza exemplos ChatML.  Trunca/padda pra max_length."""
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        input_ids = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)
        # Labels = input_ids, mas pad tokens são -100 (loss ignora)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }

# Split train/val
random.shuffle(formatted)
split = int(0.95 * len(formatted))
train_texts, val_texts = formatted[:split], formatted[split:]
print(f'Train: {len(train_texts)}, Val: {len(val_texts)}')

train_ds = ChatMLDataset(train_texts, tokenizer, MAX_LEN)
val_ds = ChatMLDataset(val_texts, tokenizer, MAX_LEN)

# Batch size cuidadoso: 2-4 em T4 com max_len=512-1024
BATCH_SIZE = 2 if DEVICE.type == 'cuda' else 1
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

print(f'Batch size: {BATCH_SIZE}')
print(f'Batches por epoch: {len(train_loader)}')


## 8. Treino — 2 epochs com AdamW + scheduler linear


In [ ]:
EPOCHS = 2
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.1

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)

print(f'Total steps: {total_steps}')
print(f'Warmup steps: {int(total_steps * WARMUP_RATIO)}')
print(f'LR inicial: {LEARNING_RATE}')


In [ ]:
train_losses = []
val_losses = []
step = 0

for epoch in range(EPOCHS):
    # ── Treino ──
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS} [train]')
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        train_losses.append(loss.item())
        step += 1
        pbar.set_postfix(loss=f'{loss.item():.3f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    avg_train = epoch_loss / max(1, len(train_loader))

    # ── Validação ──
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch + 1}/{EPOCHS} [val]'):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss_sum += outputs.loss.item()
    avg_val = val_loss_sum / max(1, len(val_loader))
    val_losses.append((step, avg_val))
    print(f'\nEpoch {epoch + 1}: train_loss={avg_train:.3f}, val_loss={avg_val:.3f}')


## 9. Curva de loss


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, alpha=0.5, label='train (batch loss)')
# Smooth com janela móvel
window = max(1, len(train_losses) // 20)
if len(train_losses) > window:
    smoothed = [sum(train_losses[max(0, i - window):i + 1]) / min(i + 1, window) for i in range(len(train_losses))]
    plt.plot(smoothed, label=f'train (smoothed, window={window})', linewidth=2)

for step, vl in val_losses:
    plt.axvline(step, color='gray', linestyle=':', alpha=0.5)
    plt.scatter([step], [vl], color='red', s=80, label=f'val (epoch end)' if step == val_losses[0][0] else None, zorder=5)

plt.xlabel('step')
plt.ylabel('loss')
plt.title('Smoke train — BR-TaxQA-R em gpt2-small-portuguese')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('oxta_contabil_smoke_loss.png', dpi=120)
plt.show()
print('Loss salva em: oxta_contabil_smoke_loss.png')


## 10. Geração DEPOIS do treino — comparação direta

Mesmas 3 perguntas. Se o treino funcionou, as respostas vão mencionar termos fiscais reais (IRPF, RFB, exercício, deduções específicas) em vez de balbuciar.


In [ ]:
model.eval()
for q in TEST_QUESTIONS:
    print(f'\n{"=" * 70}')
    print(f'PERGUNTA: {q}')
    print("=" * 70)
    print(f'\n[ANTES do treino]')
    print(f'  {before_answers[q][:300]}')
    print(f'\n[DEPOIS do treino]')
    after = generate_answer(model, tokenizer, q)
    print(f'  {after[:300]}')


## 11. Salvar modelo treinado


In [ ]:
OUT_DIR = Path('artifacts/models/oxta_contabil_smoke')
OUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# Tamanho final do checkpoint
total_mb = sum(f.stat().st_size for f in OUT_DIR.rglob('*') if f.is_file()) / 1024**2
print(f'Modelo salvo em: {OUT_DIR}')
print(f'Tamanho:         {total_mb:.1f}MB')
print(f'\nArquivos:')
for f in sorted(OUT_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name}: {f.stat().st_size / 1024**2:.2f}MB')


## 12. O que isso prova (e o que NÃO prova)

### ✓ Provado:

- Os dados BR-TaxQA-R carregam, fazem sentido, têm o conteúdo esperado
- A formatação ChatML está consistente
- Um modelo PT-BR pequeno aprende com esses dados (loss desce)
- Tokenizer + dataset pipeline funcionam end-to-end
- Geração antes/depois mostra diferença visível na qualidade

### ✗ Ainda não provado (próximos passos):

- **Que GPT-2 small em PT-BR é o modelo certo.** Não é — é só baseline.
  O alvo real é Oxta C++ 1B com 1.58-bit (sem PyTorch, sem GPU obrigatória).
- **Que 478 Q&A é suficiente.** Não é. Precisamos 5k-10k via:
  - Augment do BR-TaxQA-R existente (paraphrasing)
  - Sintético via GPT-4o-mini com manual SPED como contexto
  - BACEN FAQ (~2k adicionais)
  - Real (design partners)
- **Que tool calling funciona.** Não testado aqui. Próximo notebook.
- **Que o modelo evita alucinação de CNPJ/valor.** Não testado.
  Em produção isso vem do tool calling (não confiar em gerar números).

### Próxima coisa a fazer:

Se este notebook rodou até o fim e a curva de loss desceu, **isso é suficiente**
pra prosseguir. O próximo passo é decidir entre:

1. **Aumentar dataset** (gerar 5k sintéticos via GPT-4) — antes de escalar
2. **Escalar modelo** (40M Oxta C++ → 400M) — antes de mais dados
3. **Tool calling MVP** (formato JSON + exemplos seed) — independente dos dois

Recomendação: começar pelo **#3** porque é o que prova o conceito ao primeiro contador.
